# CSC790 Notebook 2 (Kaggle version): the 8 remaining runs

This runs only what Colab could not finish: **mBERT seeds 44, 45, 46** and **AfroXLMR-large seeds 42 to 46**. Settings are identical to the Colab notebook.

**How to use it on Kaggle:**
1. Right-hand panel → **Session options** → Accelerator: **GPU T4 x2** → Internet: **On**.
2. Top right → **Save Version** → **Save & Run All (Commit)** → **Save**. It runs in the background (about 5 to 7 hours); you can close the browser.
3. When the version shows **Successful**, open it → **Output** → download `CSC790_kaggle_results.zip`.

Only one GPU is used on purpose, so the batch size matches the Colab runs exactly.

## 0. Setup

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use exactly one GPU (same effective batch as Colab)
import re, csv, json, time, random, hashlib, shutil, unicodedata, glob
from collections import Counter
import numpy as np, pandas as pd, torch

# Where results are saved: Google Drive on Colab, the working folder elsewhere (e.g. Kaggle)
if os.path.exists("/kaggle/working"):          # Kaggle
    OUT = "/kaggle/working/CSC790"
else:
    try:                                          # Google Colab
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = "/content/drive/MyDrive/CSC790"
    except Exception:                             # anywhere else
        OUT = "CSC790_results"
for sub in ["preds", "runs"]:
    os.makedirs(f"{OUT}/{sub}", exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
print("Saving results to:", OUT, "| Device:", GPU)
if DEVICE == "cpu":
    print("WARNING: no GPU. Go to Runtime > Change runtime type > T4 GPU, then Run all again.")

## 1. Settings (you normally don't need to change anything here)

In [ ]:
MODELS_TO_RUN = ["mbert", "afroxlmr_large"]
RUN_ONLY = {("mbert", 44), ("mbert", 45), ("mbert", 46)} | {("afroxlmr_large", s) for s in [42, 43, 44, 45, 46]}
SEEDS = [42, 43, 44, 45, 46]
LABELS = ["negative", "neutral", "positive"]
l2i = {l: i for i, l in enumerate(LABELS)}

TRANSFORMERS = {
    "afriberta_large": dict(hf="castorini/afriberta_large",               lr=5e-5, bs=32, accum=1),
    "xlmr_base":       dict(hf="FacebookAI/xlm-roberta-base",             lr=2e-5, bs=32, accum=1),
    "mbert":           dict(hf="google-bert/bert-base-multilingual-cased", lr=5e-5, bs=32, accum=1),
    "afroxlmr_large":  dict(hf="Davlan/afro-xlmr-large",                  lr=1e-5, bs=16, accum=2),
}
RNNS = {
    "lstm":        dict(bidirectional=False, attention=False),
    "bilstm_attn": dict(bidirectional=True,  attention=True),
}

## 2. Load the data and fix the dev set

Same data and tweet IDs as Notebook 1. The dev set has **270 tweets that also appear in the training set**, which inflates dev scores, so we remove them before using dev to pick the best epoch.

In [ ]:
BASE = "https://raw.githubusercontent.com/afrisenti-semeval/afrisent-semeval-2023/main/data/yor"

def load(split):
    d = pd.read_csv(f"{BASE}/{split}.tsv", sep="\t", quoting=csv.QUOTE_NONE, on_bad_lines="skip")
    d["split"] = split
    return d

df = pd.concat([load(s) for s in ["train", "dev", "test"]], ignore_index=True)
df["tweet"] = df["tweet"].astype(str).str.replace('"', '', regex=False).str.strip()
df["id"] = [f"yor_{i:05d}" for i in range(len(df))]
df["text_md5"] = df.tweet.map(lambda t: hashlib.md5(t.encode("utf-8")).hexdigest())

def norm(t):
    t = t.lower()
    t = re.sub(r"@user|https?\S+|\brt\b", "", t)
    return re.sub(r"\s+", " ", t).strip()

df["norm"] = df.tweet.map(norm)
train_norm = set(df.loc[df.split == "train", "norm"])
df["dup_of_train"] = (df.split != "train") & df.norm.isin(train_norm)

train = df[df.split == "train"].reset_index(drop=True)
dev   = df[(df.split == "dev") & ~df.dup_of_train].reset_index(drop=True)
test  = df[df.split == "test"].reset_index(drop=True)
print(f"train {len(train)} | dev {len(dev)} (removed {int(((df.split=='dev') & df.dup_of_train).sum())} duplicates) | test {len(test)}")
print("test tweets also in train:", int(test.dup_of_train.sum()))
df[["id", "split", "text_md5", "dup_of_train"]].to_csv(f"{OUT}/data_index.csv", index=False)

## 3. Shared helpers

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

def scores(gold, pred):
    return {"weighted_f1": f1_score(gold, pred, average="weighted"),
            "macro_f1": f1_score(gold, pred, average="macro"),
            "accuracy": accuracy_score(gold, pred)}

def set_all_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def save_run(name, seed, probs, extra_cols, info):
    pred = [LABELS[i] for i in probs.argmax(1)]
    out = pd.DataFrame({"id": test.id, "label": test.label, "pred": pred,
                        "p_negative": probs[:, 0], "p_neutral": probs[:, 1], "p_positive": probs[:, 2]})
    for k, v in extra_cols.items():
        out[k] = v
    out.to_csv(f"{OUT}/preds/{name}_seed{seed}_test.csv", index=False)
    info.update({f"test_{k}": v for k, v in scores(test.label, pred).items()})
    info.update({"model": name, "seed": seed, "gpu": GPU,
                 "torch": torch.__version__})
    json.dump(info, open(f"{OUT}/runs/{name}_seed{seed}.json", "w"), indent=2)
    print(f"  {name} seed {seed}: test weighted F1 = {info['test_weighted_f1']:.3f} "
          f"(best epoch {info['best_epoch']}, {info['train_minutes']:.1f} min)")

def done(name, seed):
    return os.path.exists(f"{OUT}/runs/{name}_seed{seed}.json")

## 4. LSTM and BiLSTM with attention (trained from scratch)

Words are split on spaces and punctuation, keeping Yorùbá tone marks and emojis. Words seen fewer than 2 times in training become "unknown". Settings: 200-dimension word vectors, 128 hidden units, dropout 0.3, Adam (learning rate 0.001), batch 64, up to 20 epochs, stopping after 3 epochs without improvement on the cleaned dev set.

In [ ]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

TOKEN_RE = re.compile(r"[\w\u0300-\u036f']+|[^\w\s]")
MAXLEN = 64

def tokenize(t):
    t = unicodedata.normalize("NFC", t.lower())
    t = re.sub(r"https?\S+", " URL ", t)
    return TOKEN_RE.findall(t)[:MAXLEN] or ["<unk>"]

counts = Counter(tok for t in train.tweet for tok in tokenize(t))
itos = ["<pad>", "<unk>"] + sorted(w for w, c in counts.items() if c >= 2)
stoi = {w: i for i, w in enumerate(itos)}
print("Vocabulary size:", len(itos))

def encode(texts):
    ids = [[stoi.get(w, 1) for w in tokenize(t)] for t in texts]
    lens = torch.tensor([len(x) for x in ids])
    X = torch.zeros(len(ids), MAXLEN, dtype=torch.long)
    for i, x in enumerate(ids):
        X[i, :len(x)] = torch.tensor(x)
    return X, lens

class RNNClassifier(nn.Module):
    def __init__(self, vocab, bidirectional, attention, emb=200, hid=128, n_classes=3, dropout=0.3):
        super().__init__()
        self.attention, self.bidirectional = attention, bidirectional
        self.emb = nn.Embedding(vocab, emb, padding_idx=0)
        self.rnn = nn.LSTM(emb, hid, batch_first=True, bidirectional=bidirectional)
        d = hid * (2 if bidirectional else 1)
        if attention:
            self.W = nn.Linear(d, d)
            self.v = nn.Linear(d, 1, bias=False)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(d, n_classes)

    def forward(self, x, lens):
        e = self.drop(self.emb(x))
        packed = pack_padded_sequence(e, lens.cpu(), batch_first=True, enforce_sorted=False)
        out, (h, _) = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        w = None
        if self.attention:
            s = self.v(torch.tanh(self.W(out))).squeeze(-1).masked_fill(x == 0, -1e9)
            w = torch.softmax(s, dim=1)
            rep = (w.unsqueeze(-1) * out).sum(1)
        else:
            rep = torch.cat([h[-2], h[-1]], 1) if self.bidirectional else h[-1]
        return self.fc(self.drop(rep)), w

Xtr, Ltr = encode(train.tweet); ytr = torch.tensor(train.label.map(l2i).values)
Xdv, Ldv = encode(dev.tweet)
Xte, Lte = encode(test.tweet)

def predict_rnn(model, X, L, bs=256):
    model.eval(); P, W = [], []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            logits, w = model(X[i:i+bs].to(DEVICE), L[i:i+bs])
            P.append(torch.softmax(logits, 1).cpu())
            if w is not None: W.append(w.cpu())
    return torch.cat(P).numpy(), (torch.cat(W) if W else None)

def run_rnn(name, seed, max_epochs=20, patience=3, bs=64):
    set_all_seeds(seed)
    model = RNNClassifier(len(itos), **RNNS[name]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    lossf = nn.CrossEntropyLoss()
    best, best_state, best_ep, bad, t0 = -1, None, 0, 0, time.time()
    g = torch.Generator().manual_seed(seed)
    for ep in range(1, max_epochs + 1):
        model.train()
        for idx in torch.randperm(len(Xtr), generator=g).split(bs):
            logits, _ = model(Xtr[idx].to(DEVICE), Ltr[idx])
            loss = lossf(logits, ytr[idx].to(DEVICE))
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        p, _ = predict_rnn(model, Xdv, Ldv)
        f = f1_score(dev.label.map(l2i), p.argmax(1), average="weighted")
        if f > best:
            best, best_ep, bad = f, ep, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    probs, W = predict_rnn(model, Xte, Lte)
    extra = {}
    if W is not None:   # top 3 words the attention focused on, for error analysis
        toks = [tokenize(t) for t in test.tweet]
        extra["top_attention"] = [" | ".join(np.array(tk)[np.argsort(-W[i, :len(tk)].numpy())[:3]])
                                  for i, tk in enumerate(toks)]
    save_run(name, seed, probs, extra, {"best_epoch": best_ep, "dev_weighted_f1": best,
                                        "train_minutes": (time.time() - t0) / 60})

## 5. Transformer models (fine-tuned)

Max length 128, up to 10 epochs, 10% warm-up, stopping after 2 epochs without improvement on the cleaned dev set. Learning rates follow the published AfriSenti/NaijaSenti setups; AfroXLMR-large uses a smaller rate and batch 16 × 2 accumulation steps so it fits a free GPU.

In [ ]:
import transformers
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, TrainerCallback, set_seed)

class KeepBestInMemory(TrainerCallback):
    # Early stopping (patience 2) and keeping the best epoch's weights in memory.
    # Same selection rule as Hugging Face's EarlyStoppingCallback with load_best_model_at_end,
    # but without writing a full checkpoint to disk every epoch (much faster for big models).
    def __init__(self, patience=2):
        self.patience, self.best, self.bad, self.state_dict = patience, -1.0, 0, None
    def on_evaluate(self, args, state, control, metrics=None, model=None, **kw):
        f = metrics["eval_weighted_f1"]
        if f > self.best:
            self.best, self.bad = f, 0
            self.state_dict = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
        else:
            self.bad += 1
            if self.bad >= self.patience:
                control.should_training_stop = True

def run_transformer(name, seed):
    cfg = TRANSFORMERS[name]
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(cfg["hf"])
    tok.model_max_length = 512
    def to_ds(d):
        x = d[["tweet"]].copy(); x["labels"] = d.label.map(l2i).values
        return Dataset.from_pandas(x.reset_index(drop=True)).map(
            lambda b: tok(b["tweet"], truncation=True, max_length=128), batched=True, remove_columns=["tweet"])
    tr, dv, te = to_ds(train), to_ds(dev), to_ds(test)
    model = AutoModelForSequenceClassification.from_pretrained(cfg["hf"], num_labels=3, torch_dtype=torch.float32)
    tmp = f"/tmp/ckpt_{name}_{seed}"
    steps_per_epoch = -(-len(train) // (cfg["bs"] * cfg["accum"]))
    warmup = int(0.1 * 10 * steps_per_epoch)   # 10% of the 10-epoch budget
    args = TrainingArguments(
        output_dir=tmp, learning_rate=cfg["lr"], per_device_train_batch_size=cfg["bs"],
        gradient_accumulation_steps=cfg["accum"], per_device_eval_batch_size=64,
        num_train_epochs=10, warmup_steps=warmup, weight_decay=0.01,
        eval_strategy="epoch", save_strategy="no", logging_strategy="epoch",
        fp16=(DEVICE == "cuda"), report_to="none", seed=seed)
    def metrics(p):
        return {"weighted_f1": f1_score(p.label_ids, p.predictions.argmax(-1), average="weighted")}
    keeper = KeepBestInMemory(patience=2)
    trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=dv, processing_class=tok,
                      compute_metrics=metrics, callbacks=[keeper])
    t0 = time.time()
    trainer.train()
    trainer.model.load_state_dict(keeper.state_dict)   # go back to the best epoch
    evals = [h for h in trainer.state.log_history if "eval_weighted_f1" in h]
    best = max(evals, key=lambda h: h["eval_weighted_f1"])
    logits = trainer.predict(te).predictions
    probs = torch.softmax(torch.tensor(logits).float(), 1).numpy()
    info = {"best_epoch": int(round(best["epoch"])), "dev_weighted_f1": best["eval_weighted_f1"],
            "train_minutes": (time.time() - t0) / 60, "hf_model": cfg["hf"], "lr": cfg["lr"],
            "transformers": transformers.__version__,
            "collapsed": bool(len(set(probs.argmax(1))) == 1)}
    if info["collapsed"]:
        print("  WARNING: this run predicted only one class (training failed). It is kept and flagged.")
    save_run(name, seed, probs, {}, info)
    del trainer, model; torch.cuda.empty_cache(); shutil.rmtree(tmp, ignore_errors=True)

## 6. Run everything

This is the long step. Each line printed is one finished run. Safe to re-run: finished runs are skipped.

In [ ]:
for name in MODELS_TO_RUN:
    for seed in SEEDS:
        if (name, seed) not in RUN_ONLY:
            continue
        if done(name, seed):
            print(f"  {name} seed {seed}: already done, skipping"); continue
        print(f"Training {name}, seed {seed} ...")
        if name in RNNS:
            run_rnn(name, seed)
        else:
            run_transformer(name, seed)
print("ALL RUNS FINISHED")

## 7. Results table

Average over the 5 runs, with the spread (standard deviation) after the ±. "Clean test" excludes the 12 test tweets that also appear in the training set.

In [ ]:
runs = pd.DataFrame([json.load(open(f)) for f in glob.glob(f"{OUT}/runs/*.json")])
clean_ids = set(test.loc[~test.dup_of_train, "id"])
rows = []
for _, r in runs.iterrows():
    p = pd.read_csv(f"{OUT}/preds/{r.model}_seed{r.seed}_test.csv")
    c = p[p.id.isin(clean_ids)]
    rows.append({"model": r.model, "seed": r.seed,
                 "weighted_f1": r.test_weighted_f1, "macro_f1": r.test_macro_f1, "accuracy": r.test_accuracy,
                 "clean_test_weighted_f1": f1_score(c.label, c.pred, average="weighted"),
                 "best_epoch": r.best_epoch, "minutes": r.train_minutes})
res = pd.DataFrame(rows)
order = [m for m in MODELS_TO_RUN if m in set(res.model)]
summary = (res.groupby("model")[["weighted_f1", "macro_f1", "accuracy", "clean_test_weighted_f1"]]
             .agg(lambda s: f"{100*s.mean():.1f} ± {100*s.std(ddof=1):.1f}").reindex(order))
summary.insert(0, "runs", res.groupby("model").size().reindex(order))
res.to_csv(f"{OUT}/all_runs.csv", index=False)
summary.to_csv(f"{OUT}/results_summary.csv")
summary

## Next
Send a screenshot of the results table. Notebook 3 (after you both finish labelling) combines these saved predictions with your labels to answer the research questions.

## 8. Pack the results for download
Download `CSC790_kaggle_results.zip` from the **Output** tab and send it to Claude (or upload its `runs` and `preds` files into your Google Drive `CSC790` folder).

In [ ]:
shutil.make_archive("/kaggle/working/CSC790_kaggle_results", "zip", OUT)
print("Saved: /kaggle/working/CSC790_kaggle_results.zip")
print(sorted(os.listdir(f"{OUT}/runs")))